# 05. CREATE TABLE

*Module: 2 — Data definition · Companion summary: `lectures/05-create-table.md` · SQL engine: SQLite (in-memory)*

## What you will study

- write CREATE TABLE with column name, datatype, constraints
- use PRIMARY KEY for uniqueness and NOT NULL for required fields
- avoid the classic trailing-comma syntax error

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 The lookup-table example from the course

`provinces` — tiny 2-letter codes (CHAR-style) vs full names (VARCHAR-style):
```sql
CREATE TABLE provinces (id CHAR(2) PRIMARY KEY NOT NULL, name VARCHAR(24))
```


In [ ]:
q("CREATE TABLE provinces (id TEXT PRIMARY KEY NOT NULL, name TEXT NOT NULL)")

## 🧪 The full AUTHOR table from the course, with constraints

In [ ]:
q("""
CREATE TABLE author (
    author_id TEXT PRIMARY KEY NOT NULL,
    lastname  TEXT NOT NULL,
    firstname TEXT NOT NULL,
    email     TEXT,
    city      TEXT,
    country   TEXT
)
""")
q("""
INSERT INTO author (author_id, lastname, firstname, email, city, country)
VALUES ('A1', 'Chong', 'Raul', 'RFC@ca.ibm.com', 'Toronto', 'CA')
""")
q("SELECT * FROM author")

## 🧪 The constraints, firing live

In [ ]:
# Some course-syntax errors are lessons themselves — run and read the error.
import sqlite3
try:
    conn.execute("CREATE TABLE author (author_id TEXT PRIMARY KEY, lastname TEXT)")
except sqlite3.Error as e:
    print(repr(e))   # a table name can only be created once

In [ ]:
# Some course-syntax errors are lessons themselves — run and read the error.
import sqlite3
try:
    conn.execute("INSERT INTO author VALUES ('A1', 'Faker', 'Fake', 'x@x.com', 'XCity', 'XX')")
except sqlite3.Error as e:
    print(repr(e))   # PRIMARY KEY refuses duplicate author_id

In [ ]:
# Some course-syntax errors are lessons themselves — run and read the error.
import sqlite3
try:
    conn.execute("INSERT INTO author VALUES ('A5', null, 'NoName')")
except sqlite3.Error as e:
    print(repr(e))   # NOT NULL refuses a missing lastname

## 💡 Tips & tricks

- The **last column has no trailing comma** — the classic beginner CREATE error.
  ```sql
  -- ❌ trailing comma: id CHAR(2), name VARCHAR(24),
  -- ✅ id CHAR(2) PRIMARY KEY NOT NULL, name VARCHAR(24)
  ```
- Designate the primary key at creation time; retrofitting uniqueness into a populated
  table is painful.
- 📕 Dialect note — SQLite ignores VARCHAR(n) length limits (they're advisory); DB2, MySQL
  and Oracle enforce them.

## 🎯 Your turn

1. CREATE `student (student_id, name, email)` with the right two constraints so that
   ids are unique and names are mandatory.
2. Prove both constraints fire with your own INSERT attempts.

In [ ]:
# 💡 Solution
q("CREATE TABLE student (student_id TEXT PRIMARY KEY NOT NULL, name TEXT NOT NULL, email TEXT)")
q("INSERT INTO student VALUES ('S1', 'Dup', 'x@y.z')")   # this one lands
try: conn.execute("INSERT INTO student VALUES ('S1', 'Dup2', 'x@y.z')")
except sqlite3.IntegrityError as e: print('dup id:', e)
try: conn.execute("INSERT INTO student VALUES ('S2', null, 'x@y.z')")
except sqlite3.IntegrityError as e: print('no name:', e)
q("UPDATE student SET name = 'You' WHERE student_id = 'S1'")
q("SELECT * FROM student")